# Modeling — Step 1: Feature Engineering Prototype

Building the training dataset: one row per (player, season), where the
target is that season's fantasy points, and every feature is derived only
from information available *before* that season started (prior-season
stats, career/draft info). ADP is deliberately excluded as a feature --
it's the benchmark we're trying to beat, not an input.


In [59]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

seasonal = pd.read_parquet('../data/raw/seasonal_player_stats.parquet')
rosters = pd.read_parquet('../data/raw/seasonal_rosters.parquet')

print(f"seasonal: {seasonal.shape}")
print(f"rosters: {rosters.shape}")


seasonal: (6098, 58)
rosters: (30050, 37)


## Step 1: build the base table (player-season + position + target)

`seasonal` has no name/position column -- need to bring position in from
`rosters`. Using a safe-mode helper since some player-seasons have all-NaN
position values in the roster table (mode() on an all-NaN group returns
empty, which crashes a plain `.iloc[0]`).


In [60]:
SKILL_POSITIONS = ['QB', 'RB', 'WR', 'TE']

def safe_mode(x):
    m = x.mode()
    return m.iloc[0] if not m.empty else np.nan

roster_position = (
    rosters.groupby(['player_id', 'season'])['position']
    .agg(safe_mode)
    .reset_index()
)

base = seasonal.merge(roster_position, on=['player_id', 'season'], how='left')
base = base[base['position'].isin(SKILL_POSITIONS)].copy()

print(f"base: {base.shape}")
base[['player_id', 'season', 'position', 'games', 'fantasy_points_ppr']].head(10)


base: (5703, 59)


,player_id,season,position,games,fantasy_points_ppr
0,00-0007091,2015,QB,8,91.10
1,00-0010346,2015,QB,10,91.36
3,00-0019596,2015,QB,16,344.70
4,00-0019596,2016,QB,12,258.56
5,00-0019596,2017,QB,16,295.88
6,00-0019596,2018,QB,16,281.30
7,00-0019596,2019,QB,16,263.68
8,00-0019596,2020,QB,16,337.92
9,00-0019596,2021,QB,17,374.74
10,00-0019596,2022,QB,17,271.66


## Step 2: the simplest possible lag feature -- prior season's points

For each player-season, look up that SAME player's fantasy_points_ppr from
season-1.


In [61]:
prior_season = base[['player_id', 'season', 'fantasy_points_ppr', 'games']].copy()
prior_season['season'] = prior_season['season'] + 1
prior_season = prior_season.rename(columns={
    'fantasy_points_ppr': 'prior_season_points_ppr',
    'games': 'prior_season_games',
})

features = base.merge(prior_season, on=['player_id', 'season'], how='left')

print(f"features: {features.shape}")
print(f"Rows with a prior season on record: {features['prior_season_points_ppr'].notna().sum()}")
print(f"Rows with NO prior season: {features['prior_season_points_ppr'].isna().sum()}")

features[['player_id', 'season', 'position', 'fantasy_points_ppr', 'prior_season_points_ppr', 'prior_season_games']].head(10)


features: (5703, 61)
Rows with a prior season on record: 3758
Rows with NO prior season: 1945


,player_id,season,position,fantasy_points_ppr,prior_season_points_ppr,prior_season_games
0,00-0007091,2015,QB,91.10,NaN,NaN
1,00-0010346,2015,QB,91.36,NaN,NaN
2,00-0019596,2015,QB,344.70,NaN,NaN
3,00-0019596,2016,QB,258.56,344.70,16.0
4,00-0019596,2017,QB,295.88,258.56,12.0
5,00-0019596,2018,QB,281.30,295.88,16.0
6,00-0019596,2019,QB,263.68,281.30,16.0
7,00-0019596,2020,QB,337.92,263.68,16.0
8,00-0019596,2021,QB,374.74,337.92,16.0
9,00-0019596,2022,QB,271.66,374.74,17.0


## Step 3: generalize lag features to ALL numeric stats, plus a 2-season-ago lag

Rather than hand-picking a few stat columns, lag every numeric stat in
`seasonal` (targets, carries, receiving yards, target share, etc.) -- lets
XGBoost decide what's actually useful via feature importance later, rather
than us guessing up front. Also adding a season-2 lag so we can build a
trend feature (improving vs. declining trajectory).


In [62]:
# Identify which columns are actual stats to lag (exclude IDs and season_type,
# which isn't a per-player stat).
exclude_cols = ['player_id', 'season', 'season_type', 'position']
stat_cols = [c for c in base.columns if c not in exclude_cols]

def build_lag(df, shift_amount, suffix):
    lagged = df[['player_id', 'season'] + stat_cols].copy()
    lagged['season'] = lagged['season'] + shift_amount
    lagged = lagged.rename(columns={c: f'{c}_{suffix}' for c in stat_cols})
    return lagged

lag1 = build_lag(base, 1, 'lag1')  # prior season
lag2 = build_lag(base, 2, 'lag2')  # two seasons ago

features = base.merge(lag1, on=['player_id', 'season'], how='left')
features = features.merge(lag2, on=['player_id', 'season'], how='left')

print(f"features: {features.shape}")
print(f"Columns added: {features.shape[1] - base.shape[1]}")


features: (5703, 169)
Columns added: 110


## Step 4: trend feature + static career features from rosters

Trend = lag1 minus lag2 (positive = improving trajectory, negative =
declining). Career features (age, years_exp, draft_number) come from
`rosters` -- draft_number is static per player (their actual NFL draft
slot), so we take it once per player_id rather than per season.


In [63]:
features['points_trend'] = features['fantasy_points_ppr_lag1'] - features['fantasy_points_ppr_lag2']

# Age/years_exp: take once per player-season (should be stable across the
# season's roster snapshots, but use safe_mode again just in case).
career_features = (
    rosters.groupby(['player_id', 'season'])[['age', 'years_exp']]
    .agg(safe_mode)
    .reset_index()
)

# draft_number is static for a player's whole career -- take the first
# non-null value per player_id across all seasons.
draft_capital = (
    rosters[rosters['draft_number'].notna()]
    .sort_values('season')
    .groupby('player_id')['draft_number']
    .first()
    .reset_index()
)

features = features.merge(career_features, on=['player_id', 'season'], how='left')
features = features.merge(draft_capital, on='player_id', how='left')

print(f"features: {features.shape}")
print(f"Missing age: {features['age'].isna().sum()}")
print(f"Missing years_exp: {features['years_exp'].isna().sum()}")
print(f"Missing draft_number (likely undrafted players): {features['draft_number'].isna().sum()}")

features[['player_id', 'season', 'position', 'fantasy_points_ppr',
          'fantasy_points_ppr_lag1', 'points_trend', 'age', 'years_exp', 'draft_number']].head(10)


features: (5703, 173)
Missing age: 0
Missing years_exp: 26
Missing draft_number (likely undrafted players): 1712


,player_id,season,position,fantasy_points_ppr,fantasy_points_ppr_lag1,points_trend,age,years_exp,draft_number
0,00-0007091,2015,QB,91.10,NaN,NaN,39.0,17.0,187.0
1,00-0010346,2015,QB,91.36,NaN,NaN,39.0,17.0,1.0
2,00-0019596,2015,QB,344.70,NaN,NaN,38.0,15.0,199.0
3,00-0019596,2016,QB,258.56,344.70,NaN,39.0,16.0,199.0
4,00-0019596,2017,QB,295.88,258.56,-86.14,40.0,17.0,199.0
5,00-0019596,2018,QB,281.30,295.88,37.32,41.0,18.0,199.0
6,00-0019596,2019,QB,263.68,281.30,-14.58,42.0,19.0,199.0
7,00-0019596,2020,QB,337.92,263.68,-17.62,43.0,20.0,199.0
8,00-0019596,2021,QB,374.74,337.92,74.24,44.0,21.0,199.0
9,00-0019596,2022,QB,271.66,374.74,36.82,45.0,22.0,199.0


## Step 5: time-based train/validation/test split

Splitting by SEASON, not randomly -- a random split would let the model
train on some 2023 players while testing on others, which leaks
season-level information (e.g. league-wide scoring trends) across the
split. Real deployment only ever has the past to learn from.

- Train: 2015-2021
- Validation: 2022-2023 (for tuning/comparing models)
- Test: 2024 (final holdout, touch this once)


In [64]:
train = features[features['season'] <= 2021].copy()
val = features[features['season'].isin([2022, 2023])].copy()
test = features[features['season'] == 2024].copy()

print(f"Train: {train.shape[0]} rows ({train['season'].min()}-{train['season'].max()})")
print(f"Val:   {val.shape[0]} rows ({val['season'].min()}-{val['season'].max()})")
print(f"Test:  {test.shape[0]} rows ({test['season'].min()})")


Train: 3986 rows (2015-2021)
Val:   1148 rows (2022-2023)
Test:  569 rows (2024)


## Step 6: naive baseline -- "predict this year = last year's points"

For rookies / players with no prior season, fall back to the position's
median points in the training set (best simple guess with zero player
history). This baseline is what your XGBoost model actually needs to beat
-- if it can't beat "just use last year's number," it's not adding value.


In [65]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

position_median = train.groupby('position')['fantasy_points_ppr'].median()

def naive_baseline_predict(df):
    preds = df['fantasy_points_ppr_lag1'].copy()
    missing = preds.isna()
    preds[missing] = df.loc[missing, 'position'].map(position_median)
    return preds

for name, split_df in [('Validation', val), ('Test', test)]:
    preds = naive_baseline_predict(split_df)
    mae = mean_absolute_error(split_df['fantasy_points_ppr'], preds)
    rmse = mean_squared_error(split_df['fantasy_points_ppr'], preds) ** 0.5
    print(f"{name} -- Naive baseline MAE: {mae:.2f}, RMSE: {rmse:.2f}")


Validation -- Naive baseline MAE: 45.67, RMSE: 64.59
Test -- Naive baseline MAE: 49.59, RMSE: 69.40


## Step 7: train the XGBoost model

Critical leakage check first: `features` contains BOTH the lagged columns
(`_lag1`, `_lag2` -- safe, past-only) AND the original un-lagged
current-season stat columns from `base` (targets, carries, passing_yards,
etc. for the season we're trying to predict). Those un-lagged columns are
literally what fantasy_points_ppr is computed from -- including them as
inputs would mean the model sees the answer. Explicitly building the
feature list to only include lagged/career columns.


In [66]:
feature_cols = [c for c in features.columns if c.endswith('_lag1') or c.endswith('_lag2')]
feature_cols += ['points_trend', 'age', 'years_exp', 'draft_number']

print(f"Number of lag/career feature columns: {len(feature_cols)}")

# One-hot encode position (small cardinality categorical -- QB/RB/WR/TE).
features_encoded = pd.get_dummies(features, columns=['position'], prefix='pos')
position_dummy_cols = [c for c in features_encoded.columns if c.startswith('pos_')]
feature_cols_final = feature_cols + position_dummy_cols

print(f"Final feature count (incl. position dummies): {len(feature_cols_final)}")


Number of lag/career feature columns: 114
Final feature count (incl. position dummies): 118


In [67]:
train_enc = features_encoded[features_encoded['season'] <= 2021]
val_enc = features_encoded[features_encoded['season'].isin([2022, 2023])]
test_enc = features_encoded[features_encoded['season'] == 2024]

X_train, y_train = train_enc[feature_cols_final], train_enc['fantasy_points_ppr']
X_val, y_val = val_enc[feature_cols_final], val_enc['fantasy_points_ppr']
X_test, y_test = test_enc[feature_cols_final], test_enc['fantasy_points_ppr']

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")


X_train: (3986, 118), X_val: (1148, 118), X_test: (569, 118)


In [68]:
import xgboost as xgb

model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=20,
    eval_metric='mae',
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

print(f"Best iteration: {model.best_iteration}")


Best iteration: 123


In [69]:
for name, X_split, y_split in [('Validation', X_val, y_val), ('Test', X_test, y_test)]:
    preds = model.predict(X_split)
    mae = mean_absolute_error(y_split, preds)
    rmse = mean_squared_error(y_split, preds) ** 0.5
    print(f"{name} -- XGBoost MAE: {mae:.2f}, RMSE: {rmse:.2f}")

print()
print("Compare to naive baseline: Val MAE 45.67 / RMSE 64.59, Test MAE 49.59 / RMSE 69.40")


Validation -- XGBoost MAE: 41.82, RMSE: 56.61
Test -- XGBoost MAE: 43.09, RMSE: 57.94

Compare to naive baseline: Val MAE 45.67 / RMSE 64.59, Test MAE 49.59 / RMSE 69.40


## Step 8: the real test -- does the model beat ADP?

Beating the naive baseline is a good sanity check, but the actual thesis of
this project is beating ADP. Joining the model's 2024 test predictions with
that season's ADP data, then comparing how well each one (model vs. ADP)
ranks players against what actually happened -- same metric (Spearman
correlation) we used in the original EDA baseline, so it's directly
comparable.


In [70]:
adp_with_outcomes = pd.read_parquet('../data/processed/adp_with_outcomes.parquet')

model_test_results = pd.DataFrame({
    'player_id': features.loc[X_test.index, 'player_id'],
    'season': features.loc[X_test.index, 'season'],
    'position': features.loc[X_test.index, 'position'],
    'actual_points': y_test.values,
    'model_pred': model.predict(X_test),
})

adp_2024 = adp_with_outcomes[adp_with_outcomes['season'] == 2024][['player_id', 'season', 'adp_avg']]

comparison = model_test_results.merge(adp_2024, on=['player_id', 'season'], how='inner')
print(f"Matched rows (model prediction + ADP available): {len(comparison)} of {len(model_test_results)}")


Matched rows (model prediction + ADP available): 542 of 569


In [71]:
from scipy.stats import spearmanr

print(f"{'Position':<10}{'n':<6}{'Model corr':<14}{'ADP corr':<12}{'Winner'}")
print("-" * 55)

for pos in ['QB', 'RB', 'WR', 'TE']:
    subset = comparison[comparison['position'] == pos]
    if len(subset) < 5:
        continue
    # model_pred: higher = better, so correlate directly (positive expected)
    model_corr, _ = spearmanr(subset['model_pred'], subset['actual_points'])
    # adp_avg: LOWER = better/earlier pick, so correlation with actual points is expected negative
    # flip sign for apples-to-apples comparison against model_corr
    adp_corr, _ = spearmanr(-subset['adp_avg'], subset['actual_points'])
    winner = 'MODEL' if model_corr > adp_corr else 'ADP'
    print(f"{pos:<10}{len(subset):<6}{model_corr:<14.3f}{adp_corr:<12.3f}{winner}")


Position  n     Model corr    ADP corr    Winner
-------------------------------------------------------
QB        78    0.722         0.803       ADP
RB        141   0.735         0.747       ADP
WR        214   0.727         0.763       ADP
TE        109   0.663         0.672       ADP


## Step 9: what's the model actually using?

Before adding more features blindly, check feature importance on the
current model -- this tells us whether it's leaning on sensible signal
already, or missing something obvious.


In [72]:
importance = pd.DataFrame({
    'feature': feature_cols_final,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)

importance.head(25)


,feature,importance
41,fantasy_points_ppr_lag1,0.136803
40,fantasy_points_lag1,0.119738
113,draft_number,0.030836
43,tgt_sh_lag1,0.019081
96,fantasy_points_ppr_lag2,0.018164
54,ppr_sh_lag1,0.017194
31,receiving_yards_after_catch_lag1,0.016258
45,yac_sh_lag1,0.014386
79,receptions_lag2,0.013363
53,yptmpa_lag1,0.011395


## Step 10: rate stats + injury history features

**Rate stats**: the model leans heavily on total prior points but barely
uses games played -- meaning it likely struggles to distinguish "not very
good" from "good but hurt half the season." Adding explicit per-game rate
features should help it disentangle talent from availability.

**Injury history**: `injuries.parquet` was ingested at the very start of
the project but never used. Adding prior-season injury designation counts
as a real risk signal instead of leaving the model to infer it indirectly.


In [73]:
# Rate stats: prior season points/production per game, guarding against
# division by zero for anyone with 0 games played.
features['pts_per_game_lag1'] = features['fantasy_points_ppr_lag1'] / features['games_lag1'].replace(0, np.nan)
features['pts_per_game_lag2'] = features['fantasy_points_ppr_lag2'] / features['games_lag2'].replace(0, np.nan)

# A few other high-value rate stats, guarded the same way.
for stat in ['targets', 'receptions', 'carries', 'receiving_yards', 'rushing_yards']:
    lag1_col = f'{stat}_lag1'
    if lag1_col in features.columns:
        features[f'{stat}_per_game_lag1'] = features[lag1_col] / features['games_lag1'].replace(0, np.nan)

print("Rate stat columns added:", [c for c in features.columns if 'per_game' in c])
features[['player_id', 'season', 'fantasy_points_ppr_lag1', 'games_lag1', 'pts_per_game_lag1']].dropna().head(10)


Rate stat columns added: ['pts_per_game_lag1', 'pts_per_game_lag2', 'targets_per_game_lag1', 'receptions_per_game_lag1', 'carries_per_game_lag1', 'receiving_yards_per_game_lag1', 'rushing_yards_per_game_lag1']


,player_id,season,fantasy_points_ppr_lag1,games_lag1,pts_per_game_lag1
3,00-0019596,2016,344.70,16.0,21.543750
4,00-0019596,2017,258.56,12.0,21.546667
5,00-0019596,2018,295.88,16.0,18.492500
6,00-0019596,2019,281.30,16.0,17.581250
7,00-0019596,2020,263.68,16.0,16.480000
8,00-0019596,2021,337.92,16.0,21.120000
9,00-0019596,2022,374.74,17.0,22.043529
12,00-0020337,2016,131.00,7.0,18.714286
14,00-0020531,2016,304.20,15.0,20.280000
15,00-0020531,2017,332.32,16.0,20.770000


In [74]:
injuries = pd.read_parquet('../data/raw/injuries.parquet')
print(f"injuries: {injuries.shape}")
print(f"columns: {injuries.columns.tolist()}")
injuries.head(5)


injuries: (54720, 16)
columns: ['season', 'game_type', 'team', 'week', 'gsis_id', 'position', 'full_name', 'first_name', 'last_name', 'report_primary_injury', 'report_secondary_injury', 'report_status', 'practice_primary_injury', 'practice_secondary_injury', 'practice_status', 'date_modified']


,season,game_type,team,week,gsis_id,position,full_name,first_name,last_name,report_primary_injury,report_secondary_injury,report_status,practice_primary_injury,practice_secondary_injury,practice_status,date_modified
0,2015.0,REG,ARI,1.0,00-0029740,TE,Ifeanyi Momah,Ifeanyi,Momah,None,None,None,Knee,None,Did Not Participate In Practice,2015-09-09 14:14:17+00:00
1,2015.0,REG,ARI,1.0,00-0027869,G,Mike Iupati,Mike,Iupati,Knee,None,Out,Knee,None,Did Not Participate In Practice,2015-09-11 13:06:02+00:00
2,2015.0,REG,ARI,1.0,00-0029638,WR,Michael Floyd,Michael,Floyd,Hand,None,Questionable,Hand,None,Limited Participation in Practice,2015-09-11 13:05:52+00:00
3,2015.0,REG,ARI,1.0,00-0027873,TE,Jermaine Gresham,Jermaine,Gresham,Hamstring,None,Questionable,Hamstring,None,Limited Participation in Practice,2015-09-11 13:05:57+00:00
4,2015.0,REG,ARI,1.0,00-0031044,TE,Troy Niklas,Troy,Niklas,Hamstring,None,Questionable,Hamstring,None,Limited Participation in Practice,2015-09-11 13:06:07+00:00


## Step 11: build injury history features

`injuries.gsis_id` matches the `player_id` format used everywhere else.
Building weekly designation counts per player-season, then shifting into a
lag1 feature. Unlike other lag features, missing here genuinely means
"zero injury history that season" (not "unknown") -- so filling NaN with 0
is correct here specifically, unlike the stat lag features.


In [75]:
injuries['season'] = injuries['season'].astype(int)
injuries_renamed = injuries.rename(columns={'gsis_id': 'player_id'})

injury_counts = injuries_renamed.groupby(['player_id', 'season']).agg(
    weeks_on_report=('week', 'nunique'),
    weeks_out=('report_status', lambda x: (x == 'Out').sum()),
    weeks_questionable=('report_status', lambda x: (x == 'Questionable').sum()),
    weeks_doubtful=('report_status', lambda x: (x == 'Doubtful').sum()),
).reset_index()

# Shift forward one season so this becomes a "prior season" (lag1) feature,
# same convention as everything else.
injury_counts['season'] = injury_counts['season'] + 1
injury_counts = injury_counts.rename(columns={
    c: f'{c}_lag1' for c in ['weeks_on_report', 'weeks_out', 'weeks_questionable', 'weeks_doubtful']
})

features = features.merge(injury_counts, on=['player_id', 'season'], how='left')

injury_lag_cols = ['weeks_on_report_lag1', 'weeks_out_lag1', 'weeks_questionable_lag1', 'weeks_doubtful_lag1']
for col in injury_lag_cols:
    features[col] = features[col].fillna(0)

print(f"features: {features.shape}")
features[['player_id', 'season'] + injury_lag_cols].sort_values('weeks_out_lag1', ascending=False).head(10)


features: (5703, 184)


,player_id,season,weeks_on_report_lag1,weeks_out_lag1,weeks_questionable_lag1,weeks_doubtful_lag1
3127,00-0033537,2022,17.0,17.0,0.0,0.0
568,00-0027942,2020,15.0,15.0,0.0,0.0
116,00-0023564,2019,11.0,10.0,1.0,0.0
428,00-0027265,2016,10.0,10.0,0.0,0.0
3434,00-0033932,2021,12.0,10.0,1.0,1.0
2159,00-0032054,2016,9.0,9.0,0.0,0.0
1025,00-0029668,2018,9.0,8.0,0.0,0.0
1748,00-0031345,2023,10.0,8.0,0.0,0.0
1024,00-0029668,2016,11.0,7.0,3.0,0.0
2443,00-0032393,2017,13.0,7.0,5.0,0.0


## Step 12: retrain with the expanded feature set

Rebuilding the feature list to include the new rate-stat and injury-history
columns, then retraining and re-running both evaluations (vs. naive
baseline, and vs. ADP) to see whether the gap closes at all.


In [76]:
rate_cols = [c for c in features.columns if 'per_game' in c]
injury_cols = ['weeks_on_report_lag1', 'weeks_out_lag1', 'weeks_questionable_lag1', 'weeks_doubtful_lag1']

feature_cols_v2 = feature_cols + rate_cols + injury_cols

features_encoded_v2 = pd.get_dummies(features, columns=['position'], prefix='pos')
feature_cols_v2_final = feature_cols_v2 + position_dummy_cols

print(f"Feature count: v1 was {len(feature_cols_final)}, v2 is {len(feature_cols_v2_final)}")

train_v2 = features_encoded_v2[features_encoded_v2['season'] <= 2021]
val_v2 = features_encoded_v2[features_encoded_v2['season'].isin([2022, 2023])]
test_v2 = features_encoded_v2[features_encoded_v2['season'] == 2024]

X_train_v2, y_train_v2 = train_v2[feature_cols_v2_final], train_v2['fantasy_points_ppr']
X_val_v2, y_val_v2 = val_v2[feature_cols_v2_final], val_v2['fantasy_points_ppr']
X_test_v2, y_test_v2 = test_v2[feature_cols_v2_final], test_v2['fantasy_points_ppr']


Feature count: v1 was 118, v2 is 129


In [77]:
model_v2 = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=20,
    eval_metric='mae',
)

model_v2.fit(
    X_train_v2, y_train_v2,
    eval_set=[(X_val_v2, y_val_v2)],
    verbose=False,
)

for name, X_split, y_split in [('Validation', X_val_v2, y_val_v2), ('Test', X_test_v2, y_test_v2)]:
    preds = model_v2.predict(X_split)
    mae = mean_absolute_error(y_split, preds)
    rmse = mean_squared_error(y_split, preds) ** 0.5
    print(f"{name} -- XGBoost v2 MAE: {mae:.2f}, RMSE: {rmse:.2f}")

print()
print("v1 for comparison: Val MAE 41.82 / RMSE 56.61, Test MAE 43.09 / RMSE 57.94")
print("Naive baseline:    Val MAE 45.67 / RMSE 64.59, Test MAE 49.59 / RMSE 69.40")


Validation -- XGBoost v2 MAE: 41.48, RMSE: 56.16
Test -- XGBoost v2 MAE: 43.23, RMSE: 57.79

v1 for comparison: Val MAE 41.82 / RMSE 56.61, Test MAE 43.09 / RMSE 57.94
Naive baseline:    Val MAE 45.67 / RMSE 64.59, Test MAE 49.59 / RMSE 69.40


In [78]:
model_v2_test_results = pd.DataFrame({
    'player_id': features.loc[X_test_v2.index, 'player_id'],
    'season': features.loc[X_test_v2.index, 'season'],
    'position': features.loc[X_test_v2.index, 'position'],
    'actual_points': y_test_v2.values,
    'model_pred': model_v2.predict(X_test_v2),
})

comparison_v2 = model_v2_test_results.merge(adp_2024, on=['player_id', 'season'], how='inner')

print(f"{'Position':<10}{'n':<6}{'Model v2 corr':<16}{'ADP corr':<12}{'Winner'}")
print("-" * 60)
for pos in ['QB', 'RB', 'WR', 'TE']:
    subset = comparison_v2[comparison_v2['position'] == pos]
    if len(subset) < 5:
        continue
    model_corr, _ = spearmanr(subset['model_pred'], subset['actual_points'])
    adp_corr, _ = spearmanr(-subset['adp_avg'], subset['actual_points'])
    winner = 'MODEL' if model_corr > adp_corr else 'ADP'
    print(f"{pos:<10}{len(subset):<6}{model_corr:<16.3f}{adp_corr:<12.3f}{winner}")


Position  n     Model v2 corr   ADP corr    Winner
------------------------------------------------------------
QB        78    0.723           0.803       ADP
RB        141   0.731           0.747       ADP
WR        214   0.716           0.763       ADP
TE        109   0.678           0.672       MODEL


## Step 13: hyperparameter tuning

Random search over the v2 feature set. Using the validation set (2022-2023)
to pick the best combo -- NOT the test set, to keep 2024 as a genuinely
untouched final check. Each candidate trains with early stopping against
validation, same as before.


In [79]:
import random

random.seed(42)

param_distributions = {
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5, 10],
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [0.5, 1.0, 2.0, 5.0],
}

N_TRIALS = 30
results = []

for trial in range(N_TRIALS):
    params = {k: random.choice(v) for k, v in param_distributions.items()}
    trial_model = xgb.XGBRegressor(
        n_estimators=500,
        random_state=42,
        early_stopping_rounds=20,
        eval_metric='mae',
        **params,
    )
    trial_model.fit(X_train_v2, y_train_v2, eval_set=[(X_val_v2, y_val_v2)], verbose=False)
    val_preds = trial_model.predict(X_val_v2)
    val_mae = mean_absolute_error(y_val_v2, val_preds)
    results.append({**params, 'val_mae': val_mae, 'best_iteration': trial_model.best_iteration})

results_df = pd.DataFrame(results).sort_values('val_mae')
results_df.head(10)


,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,reg_alpha,reg_lambda,val_mae,best_iteration
29,6,0.05,0.6,0.6,3,0.1,5.0,40.919510,115
13,5,0.03,0.7,0.9,5,1.0,5.0,41.062216,237
8,6,0.05,0.9,0.7,3,0.5,2.0,41.199210,113
4,5,0.05,0.7,0.6,5,0.0,0.5,41.278033,98
21,8,0.01,0.6,0.8,1,0.1,1.0,41.280834,461
27,4,0.10,0.9,0.6,1,0.0,5.0,41.329648,92
5,6,0.01,0.8,0.7,5,0.0,5.0,41.338323,499
15,3,0.03,0.7,0.8,1,1.0,5.0,41.355207,240
12,8,0.05,0.7,0.8,10,1.0,1.0,41.406975,94
9,4,0.05,0.6,0.9,3,0.1,1.0,41.415768,124


## Step 14: final tuned model

Training with the best hyperparameters found, then running both evaluations
one more time (naive baseline comparison, and the ADP head-to-head) on the
untouched test set.


In [80]:
best_params = results_df.iloc[0][['max_depth', 'learning_rate', 'subsample',
                                     'colsample_bytree', 'min_child_weight',
                                     'reg_alpha', 'reg_lambda']].to_dict()
best_params['max_depth'] = int(best_params['max_depth'])
best_params['min_child_weight'] = int(best_params['min_child_weight'])

print("Best params:", best_params)

model_final = xgb.XGBRegressor(
    n_estimators=500,
    random_state=42,
    early_stopping_rounds=20,
    eval_metric='mae',
    **best_params,
)
model_final.fit(X_train_v2, y_train_v2, eval_set=[(X_val_v2, y_val_v2)], verbose=False)

for name, X_split, y_split in [('Validation', X_val_v2, y_val_v2), ('Test', X_test_v2, y_test_v2)]:
    preds = model_final.predict(X_split)
    mae = mean_absolute_error(y_split, preds)
    rmse = mean_squared_error(y_split, preds) ** 0.5
    print(f"{name} -- Final model MAE: {mae:.2f}, RMSE: {rmse:.2f}")

print()
print("v2 (untuned):   Val MAE 41.48 / RMSE 56.16, Test MAE 43.23 / RMSE 57.79")
print("Naive baseline: Val MAE 45.67 / RMSE 64.59, Test MAE 49.59 / RMSE 69.40")


Best params: {'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.6, 'colsample_bytree': 0.6, 'min_child_weight': 3, 'reg_alpha': 0.1, 'reg_lambda': 5.0}
Validation -- Final model MAE: 40.92, RMSE: 56.09
Test -- Final model MAE: 43.00, RMSE: 57.96

v2 (untuned):   Val MAE 41.48 / RMSE 56.16, Test MAE 43.23 / RMSE 57.79
Naive baseline: Val MAE 45.67 / RMSE 64.59, Test MAE 49.59 / RMSE 69.40


In [81]:
model_final_test_results = pd.DataFrame({
    'player_id': features.loc[X_test_v2.index, 'player_id'],
    'season': features.loc[X_test_v2.index, 'season'],
    'position': features.loc[X_test_v2.index, 'position'],
    'actual_points': y_test_v2.values,
    'model_pred': model_final.predict(X_test_v2),
})

comparison_final = model_final_test_results.merge(adp_2024, on=['player_id', 'season'], how='inner')

print(f"{'Position':<10}{'n':<6}{'Final model corr':<18}{'ADP corr':<12}{'Winner'}")
print("-" * 62)
for pos in ['QB', 'RB', 'WR', 'TE']:
    subset = comparison_final[comparison_final['position'] == pos]
    if len(subset) < 5:
        continue
    model_corr, _ = spearmanr(subset['model_pred'], subset['actual_points'])
    adp_corr, _ = spearmanr(-subset['adp_avg'], subset['actual_points'])
    winner = 'MODEL' if model_corr > adp_corr else 'ADP'
    print(f"{pos:<10}{len(subset):<6}{model_corr:<18.3f}{adp_corr:<12.3f}{winner}")


Position  n     Final model corr  ADP corr    Winner
--------------------------------------------------------------
QB        78    0.729             0.803       ADP
RB        141   0.743             0.747       ADP
WR        214   0.720             0.763       ADP
TE        109   0.670             0.672       ADP
